# FRR flood reduction in several Colab sessions

Each session runs whole labs on its own machine, as in `topology_builders_colab.ipynb`: every router is a network namespace of the runtime running real FRR daemons (FRR pull request 24128, draft-ietf-lsr-distoptflood), every link a veth pair. Several sessions at once give the machine-to-machine spread of what flood reduction saves: the same topologies, flood reduction off and on taking turns within each session.

- **`PLAN = 'probe'`** finds what a runtime type can do: its CPUs and memory, how long setup takes, and the largest lab of the ladder that converges in every run, with the CPU time of its quiet windows (the noise) and the hypervisor's steal time.
- **`PLAN = 'batch'`** runs the labs in `LABS`. Give each session its own `SESSION` label. With `USE_DRIVE`, the records and a copy of the FRR build go to your Google Drive, so later sessions on the same system skip the build.
- Every record gets a `session` and a `runtime` field. Combine sessions with `python -m topology_builders report` on their records put together.
- Use one Google account. Colab's FAQ does not allow using several accounts to get around its limits.

Set the first cell, then run all cells (Runtime, Run all).

In [ ]:
PLAN = 'probe'        # 'probe': LADDER in order, stopping at the first lab that does not converge; 'batch': LABS
SESSION = ''          # a label added to every record; empty: the runtime type, CPUs, memory and a random suffix
USE_DRIVE = False     # True: records and the FRR build go to Google Drive; Colab then asks for access to your Drive
DRIVE_DIR = '/content/drive/MyDrive/topology-builders-sessions'
DOWNLOAD = False      # True: offer the session's files as a download at the end (without USE_DRIVE)
BUDGET_MIN = 75       # start no new lab after this many minutes of measuring
LAB_TIMEOUT_MIN = 45  # stop a lab that takes longer than this
REPEAT = 2            # runs of each variant; flood reduction off and on take turns, the order alternating
LINKS = 2             # link changes per run, each one down and then up
LADDER = [            # (name, family, arguments of topology_builders build), smallest first
    ('rr16-d4', 'random-regular', '--routers 16 --degree 4 --seed 1'),
    ('rr32-d8', 'random-regular', '--routers 32 --degree 8 --seed 1'),
    ('rr64-d8', 'random-regular', '--routers 64 --degree 8 --seed 1'),
    ('rr64-d16', 'random-regular', '--routers 64 --degree 16 --seed 1'),
    ('ls64-s16', 'leaf-spine', '--leaves 48 --spines 16'),
]
LABS = list(LADDER)   # for PLAN = 'batch': the labs of this session, in the same form

In [ ]:
# The runtime: CPUs, memory, system, accelerator, and whether a user namespace may create network devices.
import json
import os
import random
import subprocess
import time


def out(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False).stdout.strip()


def read(path):
    with open(path) as fh:
        return fh.read()


def keyvals(text, sep):
    pairs = (line.split(sep, 1) for line in text.splitlines() if sep in line)
    return {k.strip(): v.strip().strip(chr(34)) for k, v in pairs}


lscpu = keyvals(out('lscpu'), ':')
meminfo = keyvals(read('/proc/meminfo'), ':')
osr = keyvals(read('/etc/os-release'), '=')
gpu = out('nvidia-smi -L 2>/dev/null')
tpu = ' '.join(sorted(d for d in os.listdir('/dev') if d.startswith('accel') or d == 'vfio'))
userns = subprocess.run(['unshare', '-Urn', 'ip', 'link', 'add', 'tbcheck0', 'type', 'veth', 'peer', 'name',
                         'tbcheck1'], capture_output=True, check=False).returncode == 0
env = {'cpus': os.cpu_count(), 'cpu_model': lscpu.get('Model name', ''),
       'threads_per_core': lscpu.get('Thread(s) per core', ''),
       'mem_mb': int(meminfo['MemTotal'].split()[0]) // 1024, 'os': osr.get('PRETTY_NAME', ''),
       'kernel': os.uname().release, 'gpu': gpu, 'tpu': tpu,
       'cap_eff': keyvals(read('/proc/self/status'), ':').get('CapEff', ''), 'userns_veth': userns,
       'started': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
kind = 'gpu' if gpu else ('tpu' if tpu else 'cpu')
if not SESSION:
    cpus, gib, suffix = env['cpus'], round(env['mem_mb'] / 1024), random.getrandbits(16)
    SESSION = f'{kind}-{cpus}cpu-{gib}g-{suffix:04x}'
env['session'] = SESSION
print(json.dumps(env, indent=1))
if not userns:
    print('This runtime does not allow user namespaces with network devices: the labs cannot run here.')

In [ ]:
# Where this session's files go: the runtime's disk, or Google Drive with USE_DRIVE.
OUT = '/content/tb-sessions/' + SESSION
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = DRIVE_DIR + '/' + SESSION
os.makedirs(OUT, exist_ok=True)
print('records go to', OUT)

In [ ]:
# Set up: FRR from the pull request's commit and topology-builders (tools/colab/setup.sh). With USE_DRIVE, an FRR
# build kept on Drive for this system is unpacked first, and setup.sh then skips the build.
from importlib import metadata

FRR_REF = 'e263c441160c7cf25749a9695978ffe5649678f9'
ubuntu = osr.get('VERSION_ID', 'unknown')
CACHE = f'{DRIVE_DIR}/frr-cache/ubuntu{ubuntu}-{FRR_REF[:8]}.tar.gz'
KEEP = ['/usr/lib/frr', '/usr/bin/vtysh', '/etc/ld.so.conf.d/frr.conf', '/usr/lib/pkgconfig/libyang.pc',
        '/usr/lib/x86_64-linux-gnu/pkgconfig/libyang.pc', '/usr/include/libyang', '/usr/share/yang']
t0 = time.time()
subprocess.run('git clone -q --depth 1 https://github.com/ahb-sjsu/topology-builders /content/topology-builders'
               ' 2>/dev/null || git -C /content/topology-builders pull -q', shell=True, check=False)
env['frr_from_cache'] = bool(USE_DRIVE and os.path.exists(CACHE))
if env['frr_from_cache']:
    subprocess.run(['tar', '-xzf', CACHE, '-C', '/'], check=True)
    subprocess.run(['ldconfig'], check=False)
p = subprocess.run(['bash', '/content/topology-builders/tools/colab/setup.sh'], capture_output=True, text=True,
                   check=False)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
env['setup_s'] = round(time.time() - t0)
env['setup_exit'] = p.returncode
try:
    env['topology_builders'] = metadata.version('topology-builders')
    env['networkx'] = metadata.version('networkx')
except metadata.PackageNotFoundError as e:
    env['versions_error'] = repr(e)
with open(OUT + '/env.json', 'w') as fh:
    json.dump(env, fh, indent=1)
if p.returncode != 0:
    raise RuntimeError('setup failed: see the output above')
if USE_DRIVE and not env['frr_from_cache']:
    libs = out('ls -d /usr/lib/libyang.so* /usr/lib/x86_64-linux-gnu/libyang.so* 2>/dev/null').split()
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    subprocess.run(['tar', '-czf', CACHE, *[x for x in KEEP if os.path.exists(x)], *libs], check=True)
print('setup took', env['setup_s'], 's')

In [ ]:
# The labs: for each, build the topology, write its off and on labs, and measure them in a user namespace of their
# own (unshare -Urnm), so nothing is added to the runtime's own network. One summary line per lab.
from topology_builders.measure import net_isisd_ms

TB = 'python3 -m topology_builders'
os.makedirs('/content/work/topos', exist_ok=True)
os.chdir('/content/work')
RECORDS = OUT + '/records.jsonl'
summaries = []
t_plan = time.time()


def ticks():  # steal time and total time of all CPUs, from /proc/stat
    with open('/proc/stat') as fh:
        v = [int(x) for x in fh.readline().split()[1:]]
    return (v[7] if len(v) > 7 else 0), sum(v)


def lab_summary(recs):
    s = {'runs': len(recs), 'converged': sum(1 for r in recs if r.get('converged')), 'outcomes': {}}
    for r in recs:
        for p in r['phases']:
            s['outcomes'][p['outcome']] = s['outcomes'].get(p['outcome'], 0) + 1
    for flag, fr in ((False, 'off'), (True, 'on')):
        phases = [p for r in recs if bool(r['settings']['flood_reduction']) == flag for p in r['phases']]
        for ph in ('bring-up', 'quiet', 'link-down', 'link-up'):
            ps = [p for p in phases if p['phase'] == ph]
            if not ps:
                continue
            s[ph + ' lsps ' + fr] = round(sum(p['counters'].get('tx_l2_lsp', 0) for p in ps) / len(ps), 1)
            if ph.startswith('link'):
                ms = [x for x in map(net_isisd_ms, ps) if x is not None]  # none for a window with a router unread
            else:
                ms = [p['cpu_ms'].get('isisd', 0) for p in ps]
            s[ph + ' isisd_ms ' + fr] = [round(x) for x in ms]
    return s


for name, family, args in (LADDER if PLAN == 'probe' else LABS):
    if time.time() - t_plan > BUDGET_MIN * 60:
        print('budget of', BUDGET_MIN, 'min reached; not starting', name)
        break
    topo = f'topos/{name}.json'
    subprocess.run(f'{TB} build {family} {args} --out {topo}', shell=True, check=True)
    subprocess.run(f'rm -rf labs/{name} && {TB} clab {topo} --out labs/{name} --name {name} > /dev/null',
                   shell=True, check=True)
    with open(topo) as fh:
        g = json.load(fh)
    work = f'/content/work/{name}.jsonl'
    if os.path.exists(work):
        os.remove(work)
    cmd = (f'unshare -Urnm {TB} measure labs/{name}/off labs/{name}/on --backend netns --profile --links {LINKS} '
           f'--repeat {REPEAT} --max-wait 300 --out {work} > {OUT}/{name}.log 2>&1')
    st0, tt0 = ticks()
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, start_new_session=True)
    try:
        rc = proc.wait(timeout=LAB_TIMEOUT_MIN * 60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, 9)
        rc = 'timeout'
    subprocess.run(['pkill', '-9', '-f', '/usr/lib/fr[r]/'], check=False)  # any router left behind by a stopped lab
    st1, tt1 = ticks()
    recs = []
    if os.path.exists(work):
        with open(work) as fh:
            recs = [json.loads(line) for line in fh]
    for r in recs:
        r['session'] = SESSION
        r['runtime'] = {k: env.get(k) for k in ('cpus', 'cpu_model', 'mem_mb', 'gpu', 'tpu', 'os', 'kernel')}
    with open(RECORDS, 'a') as fh:
        for r in recs:
            fh.write(json.dumps(r) + chr(10))
    s = lab_summary(recs)
    s.update({'lab': name, 'routers': len(g['nodes']), 'links': len(g.get('edges', g.get('links', []))),
              'exit': rc, 'minutes': round((time.time() - t0) / 60, 1),
              'steal_pct': round(100 * (st1 - st0) / max(1, tt1 - tt0), 2)})
    summaries.append(s)
    print(json.dumps(s))
    if PLAN == 'probe' and (rc != 0 or s['runs'] < 2 * REPEAT or s['converged'] < s['runs']):
        print('stopping the ladder:', name, 'did not converge in every run')
        break
with open(OUT + '/summary.json', 'w') as fh:
    json.dump({'env': env, 'labs': summaries}, fh, indent=1)

In [ ]:
# The session in one line (between the markers), then the package's report on its records.
print('TB-SUMMARY-BEGIN')
print(json.dumps({'env': env, 'labs': summaries}))
print('TB-SUMMARY-END')
print(out(f'{TB} report {RECORDS}'))

In [ ]:
# With USE_DRIVE the files are already on Drive. With DOWNLOAD, the session's files are packed and downloaded.
if DOWNLOAD and not USE_DRIVE:
    archive = f'/content/{SESSION}.tar.gz'
    subprocess.run(['tar', '-czf', archive, '-C', os.path.dirname(OUT), SESSION], check=True)
    from google.colab import files
    files.download(archive)
print('session', SESSION, 'files in', OUT)